# v15 — kuantisasi RR dan konsistensi pra-QRS untuk AFL konduksi bervariasi

Pra-registrasi: [RANCANGAN.md](RANCANGAN.md), dikunci 6 November 2026 **sebelum** CPSC 2021 dibuka (sha256 dan waktu di
`logs/v15_praregistrasi.log`). Penyimpangan tercatat di RANCANGAN bagian 8. Logika di `kuantisasi.py`; fitur dari
`v12_shdb_atrium/atrium.fitur_atrium(..., revisi=1)` tanpa perubahan.

Dijalankan dengan cwd = folder ini (`python tools/run_notebook.py v15_kuantisasi_rr/kuantisasi_rr.ipynb` dari root repo).

In [1]:
import json, sys, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
sys.path[:0] = ["../v11_model_tim"]
import kuantisasi as Q
from bersama import data as D, evaluasi as E
d = D.muat()

## 1. Syarat data CPSC 2021 (hasil utama)

In [2]:
syarat = Q.syarat_cpsc()
print(json.dumps(syarat, indent=1))
print("H1/H2 di CPSC 2021:", "dinilai" if syarat["syarat_terpenuhi"] else "TIDAK CUKUP DATA (syarat ≥ 5 subjek dan ≥ 200 window AFL)")

{
 "rekaman": 1425,
 "subjek": {
  "(AFL": 2,
  "(AFIB": 53,
  "(N": 54
 },
 "window": {
  "(AFL": 84,
  "(AFIB": 58703,
  "(N": 7116
 },
 "menit": {
  "(AFL": 26.3,
  "(AFIB": 9866.6,
  "(N": 1238.9
 },
 "subjek_afl": [
  "25",
  "79"
 ],
 "syarat_terpenuhi": false
}
H1/H2 di CPSC 2021: TIDAK CUKUP DATA (syarat ≥ 5 subjek dan ≥ 200 window AFL)


## 2. Analisis tambahan (penyimpangan tercatat): PTB-XL, lead I 100 Hz → 50 Hz

In [3]:
P = Q.fitur_ptbxl()
print("rekaman:", len(P["y"]), "| kelas [sinus, AFIB, AFL]:", np.bincount(P["y"]).tolist(), "| pasien AFL:", len(np.unique(P["pasien"][P["y"] == 2])))
pola = pd.crosstab(pd.Series(P["y"], name="kelas"), pd.Series(P["pola"], name="pola RR (0 cepat-teratur, 1 antara, 2 tidak teratur, 3 lambat-teratur, −1 tak terbaca)"), normalize="index").round(3)
pola

rekaman: 20500 | kelas [sinus, AFIB, AFL]: [18959, 1491, 50] | pasien AFL: 39


"pola RR (0 cepat-teratur, 1 antara, 2 tidak teratur, 3 lambat-teratur, −1 tak terbaca)",-1,0,1,2,3
kelas,,,,,
0,0.000,0.002,0.086,0.118,0.794
1,0.003,0.001,0.037,0.952,0.007
2,0.000,0.520,0.180,0.240,0.060


In [4]:
h1 = {f: Q.auroc_boot(P["fitur"][:, Q.A12.FITUR.index(f)], P["y"], P["pasien"]) for f in Q.FITUR_UJI}
sulit = (P["pola"] != 0) | (P["y"] == 1)
h1b = {f: Q.auroc_boot(np.where(sulit, P["fitur"][:, Q.A12.FITUR.index(f)], np.nan), P["y"], P["pasien"]) for f in Q.FITUR_UJI}
lulus = {f: bool(h1[f]["auroc"] >= 0.70 and h1[f]["ci"][0] > 0.5) for f in Q.FITUR_UJI}
display(pd.DataFrame({"H1 semua AFL": {f: f"{v['auroc']:.3f} [{v['ci'][0]:.2f}–{v['ci'][1]:.2f}]" for f, v in h1.items()},
                      "H1b AFL sulit": {f: f"{v['auroc']:.3f} [{v['ci'][0]:.2f}–{v['ci'][1]:.2f}]" for f, v in h1b.items()},
                      "lolos ambang H1": lulus}))
print({f: (v["n_afl"], v["pasien_afl"], v["n_afib"]) for f, v in h1.items()}, "(window AFL, pasien AFL, window AFIB)")

,H1 semua AFL,H1b AFL sulit,lolos ambang H1
kuantisasi,0.858 [0.77–0.93],0.748 [0.61–0.87],True
p_konsisten,0.924 [0.87–0.97],0.855 [0.77–0.93],True


{'kuantisasi': (50, 39, 1484), 'p_konsisten': (50, 39, 1484)} (window AFL, pasien AFL, window AFIB)


In [5]:
prob, ok = Q.h2_ptbxl(P)
h2p = Q.banding_pasien(P["y"][ok], prob[ok].argmax(1), P["p31"][ok].argmax(1), P["pasien"][ok])
pd.DataFrame({"gabungan": [h2p["titik_a"][0], h2p["titik_a"][1]],
              "paket v3.1": [h2p["titik_b"][0], h2p["titik_b"][1]],
              "Δ [CI], P": [f"{h2p[m][0]:+.3f} [{h2p[m][1]:+.3f}…{h2p[m][2]:+.3f}], P = {h2p[m][3]:.3f}" for m in ("Macro-F1", "AFL R")]},
             index=["Macro-F1", "recall AFL"])

,gabungan,paket v3.1,"Δ [CI], P"
Macro-F1,0.603241,0.622365,"-0.019 [-0.044…+0.006], P = 0.072"
recall AFL,0.760000,0.500000,"+0.260 [+0.143…+0.389], P = 1.000"


## 3. H2 di AFDB (fold resmi 5×5; latih di fold lain + SHDB-AF; bias cross-fit)

In [6]:
oof = Q.h2_afdb()
p31, _ = E.keputusan(d.oof_v31, d.folds, d.y)
pg, bg = E.keputusan(oof, d.folds, d.y)
tab = E.tabel_metrik(d, {"v31": p31, "gabungan": pg})
bd = E.banding(d, pg, p31)
afl = {"v31": E.afl_per_pasien(d, p31), "gabungan": E.afl_per_pasien(d, pg)}
display(pd.DataFrame({k: {m: f"{v[0]:.3f} [{v[1]:.2f}–{v[2]:.2f}]" for m, v in tab[k].items() if m in ("Macro-F1","AFL F1","AFL R","AFL P","Normal R")} for k in tab}))
display(pd.DataFrame({m: f"Δ {v[0]:+.3f} [{v[1]:+.3f}…{v[2]:+.3f}], P = {v[3]:.3f} → {E.vonis(v[3])}" for m, v in bd.items()}, index=["gabungan vs v3.1"]).T)
display(pd.DataFrame(afl).round(2).rename_axis("recall AFL per pasien"))
print("bias median", np.median(bg.reshape(-1, 3), 0), "| di batas", E.bias_di_batas(bg))

,v31,gabungan
Macro-F1,0.832 [0.67–0.92],0.756 [0.66–0.86]
Normal R,0.924 [0.84–0.97],0.927 [0.84–0.97]
AFL P,0.715 [0.26–0.95],0.521 [0.17–0.72]
AFL R,0.592 [0.13–0.91],0.346 [0.15–0.75]
AFL F1,0.647 [0.17–0.89],0.413 [0.18–0.71]


,gabungan vs v3.1
Macro-F1,"Δ -0.076 [-0.112…+0.014], P = 0.037 → lebih buruk"
AFL F1,"Δ -0.235 [-0.335…+0.032], P = 0.032 → lebih buruk"
AFL P,"Δ -0.194 [-0.339…-0.082], P = 0.015 → lebih buruk"
AFL R,"Δ -0.245 [-0.333…+0.092], P = 0.064 → tidak te..."
Normal R,"Δ +0.003 [-0.003…+0.009], P = 0.852 → tidak te..."


,v31,gabungan
recall AFL per pasien,,
04043,0.40,0.20
04908,0.92,0.69
04936,0.51,0.16
06426,0.17,0.47
07910,0.00,0.10
08215,0.00,0.00
08378,0.96,0.78


bias median [ 0.   -0.5  -2.25] | di batas 0.0


In [7]:
stres = Q.h2_stres()                      # protokol butir 9; fitur EKG = median data latih (tidak ada EKG sintetis)
pd.DataFrame(stres).rename_axis("ambang bpm berhenti menebak Normal")

,gabungan,v31
ambang bpm berhenti menebak Normal,,
tanpa variasi,120,115
seragam ±10 ms,120,115
Gauss sd 10 ms,115,115
Gauss sd 20 ms,115,115


In [8]:
ring = {"syarat_cpsc": syarat, "h1_ptbxl": h1, "h1b_ptbxl": h1b, "h1_lulus_ptbxl": lulus, "h2_ptbxl": h2p,
        "pola_ptbxl": pola.to_dict(), "h2_afdb": {"metrik": tab, "banding_v31": bd, "afl_per_pasien": afl,
        "bias_median": np.median(bg.reshape(-1, 3), 0).tolist(), "bias_di_batas": E.bias_di_batas(bg)}, "stres": stres,
        "n_ptbxl": {"rekaman": int(len(P["y"])), "kelas": np.bincount(P["y"]).tolist(), "rr_terbaca": int(ok.sum())},
        "lingkungan": E._lingkungan()}
(Q.CACHE / "ringkasan.json").write_text(json.dumps(ring, indent=1, default=float, ensure_ascii=False))
print("tersimpan:", Q.CACHE / "ringkasan.json")

tersimpan: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v15/ringkasan.json


## 4. Kesimpulan

Dijalankan di PC lewat `tools/run_notebook.py`, 6 November 2026 (±4 menit dengan cache; fitur PTB-XL ±3 menit, tahap gabungan AFDB ±1 menit).

- **Hasil utama (pra-registrasi): tidak cukup data.** CPSC 2021 (1.425 rekaman yang dirilis) hanya memuat AFL dari 2 subjek (25 dan 79; 26,3 menit; 84 window 10 s murni). Syarat yang dikunci: ≥ 5 subjek dan ≥ 200 window. H1/H2 di CPSC 2021 tidak dinilai.
- **Analisis tambahan PTB-XL (penyimpangan tercatat, bukan hasil utama):** 20.500 rekaman, AFL 50 dari 39 pasien. Konsistensi pra-QRS AUROC AFL vs AFIB 0,924 [0,87–0,97], kuantisasi RR 0,858 [0,77–0,93] → keduanya lolos ambang H1. Pada AFL sulit (bukan cepat-teratur; 24 rekaman): 0,855 dan 0,748.
- **Tahap gabungan di PTB-XL:** recall AFL 0,50 → 0,76 (+0,26, P = 1,00); Macro-F1 0,622 → 0,603 (−0,019, P = 0,072, tidak terbedakan).
- **Tahap gabungan di AFDB (fold resmi + SHDB-AF): lebih buruk.** Macro-F1 0,756 vs 0,832 (Δ −0,076, P = 0,037); AFL F1 −0,235 (P = 0,032). Per pasien: AFL sulit membaik (06426 0,17 → 0,47; 07910 0,00 → 0,10), tetapi 04936 0,51 → 0,16 dan 08378 0,96 → 0,78.
- **Uji stres takikardia (fitur = median):** ambang 120/120/115/115 bpm vs v3.1 115; praktis sama.
- **Makna:** informasi atrium dari EKG 50 Hz membedakan AFL dari AFIB, termasuk AFL yang lolos dari model RR, tetapi penggabungan sederhana dengan tiga kelas sekaligus merusak pola AFL yang sudah tertangkap (terutama 04936). Usulan: pakai fitur ini hanya untuk memisahkan AFIB vs AFL pada window yang sudah diputuskan AF, dan uji pada rekaman perangkat dengan rancangan baru.